In [2]:
from google.colab import files
uploaded = files.upload()

Saving amazon_delivery.csv to amazon_delivery.csv


In [3]:
import pandas as pd


df = pd.read_csv('amazon_delivery.csv')


print(f"Total deliveries: {len(df)}")
print(f"\nColumns:")
for col in df.columns:
    print(f"  - {col}")
print(f"\nFirst 3 rows:")
print(df.head(3))

Total deliveries: 43739

Columns:
  - Order_ID
  - Agent_Age
  - Agent_Rating
  - Store_Latitude
  - Store_Longitude
  - Drop_Latitude
  - Drop_Longitude
  - Order_Date
  - Order_Time
  - Pickup_Time
  - Weather
  - Traffic
  - Vehicle
  - Area
  - Delivery_Time
  - Category

First 3 rows:
        Order_ID  Agent_Age  Agent_Rating  Store_Latitude  Store_Longitude  \
0  ialx566343618         37           4.9       22.745049        75.892471   
1  akqg208421122         34           4.5       12.913041        77.683237   
2  njpu434582536         23           4.4       12.914264        77.678400   

   Drop_Latitude  Drop_Longitude  Order_Date Order_Time Pickup_Time  \
0      22.765049       75.912471  2022-03-19   11:30:00    11:45:00   
1      13.043041       77.813237  2022-03-25   19:45:00    19:50:00   
2      12.924264       77.688400  2022-03-19   08:30:00    08:45:00   

      Weather Traffic      Vehicle            Area  Delivery_Time     Category  
0       Sunny   High   motorcy

In [4]:

print("=== DATA OVERVIEW ===")
print(f"\nShape: {df.shape}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nBasic stats:")
print(df.describe())

=== DATA OVERVIEW ===

Shape: (43739, 16)

Data types:
Order_ID            object
Agent_Age            int64
Agent_Rating       float64
Store_Latitude     float64
Store_Longitude    float64
Drop_Latitude      float64
Drop_Longitude     float64
Order_Date          object
Order_Time          object
Pickup_Time         object
Weather             object
Traffic             object
Vehicle             object
Area                object
Delivery_Time        int64
Category            object
dtype: object

Missing values:
Order_ID            0
Agent_Age           0
Agent_Rating       54
Store_Latitude      0
Store_Longitude     0
Drop_Latitude       0
Drop_Longitude      0
Order_Date          0
Order_Time          0
Pickup_Time         0
Weather            91
Traffic             0
Vehicle             0
Area                0
Delivery_Time       0
Category            0
dtype: int64

Basic stats:
          Agent_Age  Agent_Rating  Store_Latitude  Store_Longitude  \
count  43739.000000  43685.000000

In [6]:


df['Agent_Rating'] = df['Agent_Rating'].fillna(df['Agent_Rating'].median())
df['Weather'] = df['Weather'].fillna('Unknown')

print("✅ Clean. No warnings.")

df['Avg_Delivery'] = df['Delivery_Time'].mean()

df['Ghost_Risk'] = 'Low Risk'


df.loc[
    (df['Delivery_Time'] < df['Delivery_Time'].quantile(0.10)) &
    (df['Agent_Rating'] < 4.0),
    'Ghost_Risk'
] = 'High Risk'

df.loc[
    (df['Delivery_Time'] < df['Delivery_Time'].quantile(0.25)) &
    (df['Agent_Rating'] < 4.5),
    'Ghost_Risk'
] = 'Medium Risk'

print(df['Ghost_Risk'].value_counts())
print(f"\nHigh Risk deliveries: {len(df[df['Ghost_Risk']=='High Risk'])}")

✅ Clean. No warnings.
Ghost_Risk
Low Risk       43107
Medium Risk      632
Name: count, dtype: int64

High Risk deliveries: 0


In [7]:
medium_risk = df[df['Ghost_Risk'] == 'Medium Risk']

print(f"Medium risk deliveries: {len(medium_risk)}")
print(f"\nAverage delivery time — Medium Risk: {round(medium_risk['Delivery_Time'].mean(), 1)} mins")
print(f"Average delivery time — All: {round(df['Delivery_Time'].mean(), 1)} mins")
print(f"\nWeather breakdown:")
print(medium_risk['Weather'].value_counts())
print(f"\nArea breakdown:")
print(medium_risk['Area'].value_counts())
print(f"\nVehicle breakdown:")
print(medium_risk['Vehicle'].value_counts())

Medium risk deliveries: 632

Average delivery time — Medium Risk: 44.8 mins
Average delivery time — All: 124.9 mins

Weather breakdown:
Weather
Windy         154
Stormy        142
Sandstorms    135
Cloudy         72
Fog            72
Sunny          53
Unknown         4
Name: count, dtype: int64

Area breakdown:
Area
Metropolitian     493
Urban             124
Other              15
Name: count, dtype: int64

Vehicle breakdown:
Vehicle
motorcycle     466
scooter        134
van             31
bicycle          1
Name: count, dtype: int64


In [8]:
# Save findings
medium_risk.to_csv('ghost_risk_deliveries.csv', index=False)

summary = pd.DataFrame({
    'Metric': ['Total Deliveries', 'Suspicious Deliveries', 'Suspicious Rate',
               'Avg Time All', 'Avg Time Suspicious', 'Top Area', 'Top Vehicle'],
    'Value': [43739, 632, '1.4%', '124.9 mins', '44.8 mins',
              'Metropolitan', 'Motorcycle']
})
summary.to_csv('ghost_summary.csv', index=False)

from google.colab import files
files.download('ghost_risk_deliveries.csv')
files.download('ghost_summary.csv')

print("✅ Files saved!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Files saved!


In [9]:
import pandas as pd

summary = pd.DataFrame({
    'Metric': ['Total Deliveries', 'Suspicious Deliveries', 'Suspicious Rate',
               'Avg Time All Deliveries', 'Avg Time Suspicious',
               'Top Area', 'Top Vehicle', 'Top Weather'],
    'Value': ['43,739', '632', '1.4%', '124.9 mins', '44.8 mins',
              'Metropolitan (78%)', 'Motorcycle (74%)', 'Windy/Stormy/Sandstorm']
})

summary.to_csv('ghost_summary.csv', index=False)

from google.colab import files
files.download('ghost_summary.csv')

print("✅ Downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded!
